In [6]:
from MTSM_tools import *

gdf=load_gdf('rec').dropna(subset='rec_fl_adu')

start_time_offset=load_gdf('set').query("ID_set=='static_start_time'")['set_value'].iloc[0]
if start_time_offset !=np.nan:
	start_time_offset=start_time_offset.split(':')
	start_time_offset=pd.Timedelta(hours=float(start_time_offset[0]),minutes=float(start_time_offset[1]))
else:
	start_time_offset=pd.Timedelta(hours=0)

with open('MTSM/lib/epsg.txt') as file:
	crs=file.read()

gdf['xml_ch03_ser_num']=gdf['xml_ch03_ser_num'].replace(np.nan,0).astype(str).str.zfill(3)
gdf['xml_ch04_ser_num']=gdf['xml_ch04_ser_num'].replace(np.nan,0).astype(str).str.zfill(3)
gdf['xml_ch05_ser_num']=gdf['xml_ch05_ser_num'].replace(np.nan,0).astype(str).str.zfill(3)

gdf['adu']=gdf['rec_fl_adu']

gdf['rec_fl_rec_start']=gdf['rec_fl_rec_start'].dt.tz_localize(None)
gdf['rec_fl_rec_start']

if start_time_offset.seconds<1:
	gdf['start']=gdf['xml_rec_start']
	gdf['end']=gdf['xml_rec_end']
else:
	gdf['start']=gdf['xml_rec_start'].dt.normalize()+start_time_offset
	gdf['end']=gdf['xml_rec_end'].dt.normalize()+start_time_offset

start=[];end=[]
for st,fl,ts,te in zip(gdf['rec_qc_status'],gdf['rec_fl_rec_start'].copy().dt.normalize(),gdf['xml_rec_start'],gdf['xml_rec_end']):
	if st=='Recording':
		start.append(fl+start_time_offset)
		end.append(fl+pd.Timedelta(hours=23,minutes=59))
	else:
		start.append(ts)
		end.append(te)

gdf['start']=start
gdf['end']=end

gdf['start']=[t.round('min') for t in gdf['start']]
gdf['end']=[t.round('min') for t in gdf['end']]

gdf['y1']=gdf['start']-gdf['start'].min()
gdf['y1']=[td.total_seconds()/60 for td in gdf['y1']]
gdf['y2']=gdf['end']-gdf['start'].min()
gdf['y2']=[td.total_seconds()/60 for td in gdf['y2']]


df=pd.merge(gdf,pd.DataFrame(gdf['adu'].sort_values().unique()).reset_index(names='x').set_index(0),how='left',left_on='adu',right_index=True)
dx=100
df['x']=((df['x']+1)*10*dx)

df['txt']=df['ID_rec'].astype('str')+' - '+gdf['xml_ch03_ser_num']+', '+gdf['xml_ch04_ser_num']+', '+gdf['xml_ch05_ser_num']


geom=[]
for x,y1,y2 in zip(df['x'],df['y1'],df['y2']):
	geom.append(Polygon(((y1,x+dx),(y2,x+dx),(y2,x-dx),(y1,x-dx))))

df=df[['ID_rec','start','end','txt','adu','x','y1','y2','rec_qc_status']]

gdf=gpd.GeoDataFrame(df,geometry=geom,crs=crs)
gdf.to_file('MTSM_qgis/mtsm_timeline.gpkg',engine='pyogrio')


In [10]:
xmin=(gdf['start'].dt.normalize().min()+pd.Timedelta(days=-gdf['start'].min().day_of_week))
xmax=(gdf['end'].dt.normalize().max()+pd.Timedelta(days=(6-gdf['end'].max().day_of_week)))
ymin=gdf['x'].min()-8*dx
ymax=gdf['x'].max()+3*dx


df=pd.DataFrame(index=np.arange(xmin,xmax,timedelta(hours=1)))
df['x']=(df.index.astype('datetime64[ms]')-gdf['start'].min()).total_seconds()/60
df['dt']=df.index.astype('datetime64[ms]')
df['week']=((df['x']-df['x'].min())//(7*24*60)).astype(float)
df['week']+=1

def get_week_polygons(df):
	gb=df.groupby('week',as_index=False).agg(x1=('x','min'),x2=('x','max'),ts=('dt','min'),te=('dt','max'))

	geom=[]
	for x1,x2,week in zip(gb['x1'],gb['x2'],gb['week']):
		geom.append(Polygon(((x1,ymax),(x2+60,ymax),(x2+60,ymin),(x1,ymin))))

	gpd.GeoDataFrame(gb,geometry=geom,crs=crs).to_file('MTSM_qgis/mtsm_timeline.gpkg',layer='mtsm_timeline_weeks',engine='pyogrio')

def get_days_ls(df):
	df2=df.loc[df['dt']==df['dt'].dt.normalize()]
	df2['date']=df2['dt'].dt.date
	geom=[]
	for x in df2['x']:
		geom.append(LineString(((x,ymin),(x,ymax))))

	gpd.GeoDataFrame(df2,geometry=geom,crs=crs).to_file('MTSM_qgis/mtsm_timeline.gpkg',layer='mtsm_timeline_days',engine='pyogrio')

def get_hours_ls(df):
	geom=[]
	for x in df['x']:
		geom.append(LineString(((x,ymin),(x,ymax))))

	gpd.GeoDataFrame(df,geometry=geom,crs=crs).to_file('MTSM_qgis/mtsm_timeline.gpkg',layer='mtsm_timeline_hours',engine='pyogrio')

get_week_polygons(df)
get_days_ls(df)
get_hours_ls(df)



In [8]:
df2=df.loc[df['x']%]

SyntaxError: invalid syntax (1240114477.py, line 1)

10080